# Panel de Control de Simulaciones: Casa de Campo (Madrid)

Este notebook permite orquestar la generación de escenarios reales mediante SAREnv y la ejecución de algoritmos bioinspirados del framework MTS.

In [18]:
import os
import sys
import json
import subprocess
import pandas as pd
import numpy as np

# Cambiamos al directorio raíz del framework para que las rutas relativas funcionen correctamente
# La ubicación esperada del notebook es framwork-MTS/MTS-UncertainEnvironment-Algoritmos-bioinspirados/TFM_JC/pruebas/
current_path = os.getcwd()
if 'MTS-UncertainEnvironment-Algoritmos-bioinspirados' in current_path:
    while not os.path.exists('bf-busqueda.py'):
        os.chdir('..')

print(f"Directorio de trabajo actual: {os.getcwd()}")

Directorio de trabajo actual: c:\Users\juanc\Desktop\TFM\TFM-Juan Carlos\Software\framwork-MTS\MTS-UncertainEnvironment-Algoritmos-bioinspirados


## 1. Generación del Entorno Real (SAREnv)
Aseguramos que el mapa de la Casa de Campo y su configuración base están disponibles llamando al script puente.

In [19]:
print("Generando/Verificando escenario real desde SAREnv...")
# Configuramos la ruta al ejecutable de python del entorno virtual de la raíz
python_exe = os.path.abspath("../../tfm/Scripts/python.exe") if os.name == 'nt' else "python"

if not os.path.exists(python_exe):
    print(f"ADVERTENCIA: No se encontró el entorno virtual en {python_exe}. Usando python del sistema.")
    python_exe = "python"

result = subprocess.run([python_exe, "generar_escenario_real_jc.py"], capture_output=True, text=True)
print(result.stdout)
if result.returncode != 0:
    print("ERROR en la generación del escenario:")
    print(result.stderr)

Generando/Verificando escenario real desde SAREnv...
Generando mapa en SAREnv...
Mapa binario guardado en: c:\Users\juanc\Desktop\TFM\TFM-Juan Carlos\Software\framwork-MTS\MTS-UncertainEnvironment-Algoritmos-bioinspirados\TFM_JC/pruebas/escenario_real_casacampo.npy
Archivo de configuraciÃ³n JSON generado: c:\Users\juanc\Desktop\TFM\TFM-Juan Carlos\Software\framwork-MTS\MTS-UncertainEnvironment-Algoritmos-bioinspirados\TFM_JC/pruebas/escenario_real_casacampo.json
Dimensiones del mapa inyectadas: 119 (X) x 139 (Y)



## 2. Configuración Dinámica del Experimento
Definir las variables de la simulación aquí. El notebook actualizará el JSON automáticamente.

In [20]:
# === VARIABLES DEL EXPERIMENTO ===
algoritmo = "ACO"  # Opciones: ACO, ABC, BHA, lawnmower, expanding_sq, voraz-heur, voraz-myope
num_drones = 2
bateria_pasos = 1000
pdmax = 0.8
dmax = 2.1
sigma = 0.7

config_path = "TFM_JC/pruebas/escenario_real_casacampo.json"

with open(config_path, 'r') as f:
    data = json.load(f)

# Inyección de parámetros dinámicos
data['algoritmo_busqueda'] = algoritmo
data['num_agents'] = num_drones
data['num_steps'] = bateria_pasos
data['pdmax'] = pdmax
data['dmax'] = dmax
data['sigma'] = sigma

# Reposicionamiento básico de drones (diagonal simple para evitar colisiones iniciales)
data['init_pos'] = [[10 + i*5, 10 + i*5] for i in range(num_drones)]

with open(config_path, 'w') as f:
    json.dump(data, f, indent=4)

print(f"Configuración actualizada: {algoritmo} | Agentes: {num_drones} | Pasos: {bateria_pasos}")

Configuración actualizada: ACO | Agentes: 2 | Pasos: 1000


## 3. Ejecución de la Simulación
Lanza el proceso de búsqueda bioinspirada.

In [21]:
print(f"Iniciando simulación de búsqueda con {algoritmo}...")
result = subprocess.run([python_exe, "bf-busqueda.py", config_path], capture_output=True, text=True)
print(result.stdout)

if "OK" in result.stdout:
    print("Simulación completada con éxito.")
else:
    print("La simulación terminó con posibles errores:")
    print(result.stderr)

Iniciando simulación de búsqueda con ACO...
Cargando mapa real desde: TFM_JC/pruebas\escenario_real_casacampo.npy
EvoluciÃ³n guardada en: TFG_Romeo\resultados\funciones_obj\ACO_evolution_min_agents2_iter5_FOET_20260526_145732.csv
Resultados guardados en: resultados\escenario_real_casacampo\bf_aco_ET-4.csv
OK

Simulación completada con éxito.


## 4. Análisis de Resultados en Tiempo Real
Cargamos el CSV más reciente generado para esta configuración.

In [22]:
import glob

# Localizar el subdirectorio de resultados (basado en el nombre del JSON)
nombre_prueba = os.path.basename(config_path).replace('.json', '')
search_pattern = f"resultados/**/{nombre_prueba}/*.csv"
csv_files = glob.glob(search_pattern, recursive=True)

if csv_files:
    # Ordenar por fecha de modificación para sacar el último
    latest_csv = max(csv_files, key=os.path.getmtime)
    print(f"Analizando resultados de: {latest_csv}")
    df = pd.read_csv(latest_csv, index_col=0)
    
    # Limpieza visual de coordenadas para la tabla
    display(df.style.set_caption("Métricas de la última ejecución"))
    
    # Resumen rápido
    if not pd.isna(df.loc['Found Target', 'Target']):
        agente_exito = int(float(df.loc['Found Target', 'Target']))
        print(f"\033[92m¡ÉXITO! La víctima fue localizada por el Agente {agente_exito}.\033[0m")
    else:
        print("\033[91mFALLO: Los drones agotaron la batería sin localizar a la víctima.\033[0m")
else:
    print("No se encontraron archivos de resultados en la ruta esperada.")

Analizando resultados de: resultados\escenario_real_casacampo\bf_aco_ET-4.csv


,Agent 0,Agent 1,Target
Initial position,[10. 10.],[15. 15.],[2 2]
Final position,[3. 4.],[24. 14.],[2 2]
Distance,282.1198,281.7056,0.0
Steps taken,232,232,0
Found Target,True,False,0
Semilla,-,-,0


¡ÉXITO! La víctima fue localizada por el Agente 0.
